# Task 3: SQL for Data Analysis
**Dataset:** E-Commerce Orders | **Tool:** SQLite (via Python)

---

In [ ]:
# STEP 1: Setup - Upload data.csv first (click folder icon on left → upload)
import sqlite3, pandas as pd

df = pd.read_csv('data.csv', encoding='latin1')
df['Revenue'] = df['Quantity'] * df['UnitPrice']
conn = sqlite3.connect('ecommerce.db')
df.to_sql('orders', conn, if_exists='replace', index=False)
print('✅ Database ready! Rows loaded:', len(df))
print('Columns:', list(df.columns))

## Query 1: SELECT + WHERE + ORDER BY
Get orders from Germany with Revenue > 50

In [ ]:
result = pd.read_sql("""
SELECT InvoiceNo, Description, Quantity, UnitPrice, Revenue, Country
FROM orders
WHERE Country = 'Germany' AND Revenue > 50
ORDER BY Revenue DESC
LIMIT 10
""", conn)
print(result)

## Query 2: GROUP BY + Aggregate Functions (SUM, AVG, COUNT)
Total revenue by country

In [ ]:
result = pd.read_sql("""
SELECT Country,
       COUNT(*) AS TotalOrders,
       ROUND(SUM(Revenue), 2) AS TotalRevenue,
       ROUND(AVG(Revenue), 2) AS AvgOrderRevenue
FROM orders
GROUP BY Country
ORDER BY TotalRevenue DESC
LIMIT 10
""", conn)
print(result)

## Query 3: WHERE vs HAVING
- **WHERE** filters rows BEFORE grouping
- **HAVING** filters groups AFTER aggregation

In [ ]:
result = pd.read_sql("""
SELECT Country, ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM orders
WHERE Quantity > 0
GROUP BY Country
HAVING TotalRevenue > 100000
ORDER BY TotalRevenue DESC
""", conn)
print(result)

## Query 4: Average Revenue Per User (ARPU)

In [ ]:
result = pd.read_sql("""
SELECT ROUND(SUM(Revenue) / COUNT(DISTINCT CustomerID), 2) AS AvgRevenuePerUser
FROM orders
WHERE CustomerID IS NOT NULL
""", conn)
print(result)

## Query 5a: INNER JOIN
Top spending customers who placed more than 5 orders

In [ ]:
result = pd.read_sql("""
SELECT o.CustomerID, o.Country, c.TotalOrders,
       ROUND(SUM(o.Revenue), 2) AS TotalSpend
FROM orders o
INNER JOIN (
    SELECT CustomerID, COUNT(DISTINCT InvoiceNo) AS TotalOrders
    FROM orders
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
    HAVING TotalOrders > 5
) c ON o.CustomerID = c.CustomerID
GROUP BY o.CustomerID
ORDER BY TotalSpend DESC
LIMIT 10
""", conn)
print(result)

## Query 5b: LEFT JOIN
All products including those with no positive sales

In [ ]:
result = pd.read_sql("""
SELECT a.Description,
       COALESCE(b.TotalSold, 0) AS TotalSold
FROM orders a
LEFT JOIN (
    SELECT Description, SUM(Quantity) AS TotalSold
    FROM orders
    WHERE Quantity > 0
    GROUP BY Description
) b ON a.Description = b.Description
GROUP BY a.Description
ORDER BY TotalSold DESC
LIMIT 10
""", conn)
print(result)

## Query 6: Subquery
Orders where revenue is above the average revenue

In [ ]:
result = pd.read_sql("""
SELECT InvoiceNo, Description, ROUND(Revenue, 2) AS Revenue
FROM orders
WHERE Revenue > (
    SELECT AVG(Revenue) FROM orders WHERE Revenue > 0
)
ORDER BY Revenue DESC
LIMIT 10
""", conn)
print(result)

## Query 7: NULL Value Handling with COALESCE
Replace NULL CustomerID with 'Guest'

In [ ]:
# Count NULLs
result = pd.read_sql("""
SELECT COUNT(*) AS NullCustomers FROM orders WHERE CustomerID IS NULL
""", conn)
print('NULL CustomerIDs:', result)

# Handle with COALESCE
result2 = pd.read_sql("""
SELECT COALESCE(CustomerID, 'Guest') AS CustomerID,
       COUNT(*) AS Orders,
       ROUND(SUM(Revenue), 2) AS Revenue
FROM orders
GROUP BY COALESCE(CustomerID, 'Guest')
ORDER BY Revenue DESC
LIMIT 10
""", conn)
print(result2)

## Query 8: CREATE VIEW
Create a reusable customer revenue summary view

In [ ]:
conn.execute('DROP VIEW IF EXISTS customer_revenue_summary')
conn.execute("""
CREATE VIEW customer_revenue_summary AS
SELECT CustomerID, Country,
       COUNT(DISTINCT InvoiceNo) AS TotalInvoices,
       ROUND(SUM(Revenue), 2)    AS TotalRevenue,
       ROUND(AVG(Revenue), 2)    AS AvgOrderValue,
       MAX(InvoiceDate)          AS LastOrderDate
FROM orders
WHERE CustomerID IS NOT NULL AND Quantity > 0
GROUP BY CustomerID
""")
result = pd.read_sql("""
SELECT * FROM customer_revenue_summary
ORDER BY TotalRevenue DESC LIMIT 10
""", conn)
print(result)

## Query 9: Index for Query Optimization
Indexes speed up filtering on large tables

In [ ]:
conn.execute('CREATE INDEX IF NOT EXISTS idx_customer  ON orders(CustomerID)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_country   ON orders(Country)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_invoiceno ON orders(InvoiceNo)')
print('✅ Indexes created!')

# Fast lookup using index
result = pd.read_sql("""
SELECT CustomerID, ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM orders
WHERE CustomerID = '17850.0'
GROUP BY CustomerID
""", conn)
print(result)

## Query 10: Monthly Revenue Trend (Bonus)

In [ ]:
result = pd.read_sql("""
SELECT SUBSTR(InvoiceDate, 1, 7) AS Month,
       ROUND(SUM(Revenue), 2)    AS MonthlyRevenue,
       COUNT(DISTINCT CustomerID) AS UniqueCustomers
FROM orders
WHERE Quantity > 0
GROUP BY Month
ORDER BY MonthlyRevenue DESC
LIMIT 12
""", conn)
print(result)

---
